# Notebook 16: Mixed-condition Q-Learning

This is the final tabular Q-Learning comparison for Study 3. It tests whether training episodes sampled across low, reference, and high pollution conditions improve generalization. Reward equations, environmental dynamics, economic parameters, and the observation encoding stay fixed.

## Comparison design

- Train local, global, and mixed-reward policies with seeds 11, 23, and 37.
- Compare 300, 1,000, and 3,000 episodes.
- Train scenario-specific policies as controls and train a second set with pollution conditions sampled episode by episode.
- Use the same revised epsilon schedule for both sets. Epsilon decays from 1.0 to 0.05 across 3,000 episodes, keeping exploration active past episode 300.
- Evaluate each checkpoint under low, reference, and high pollution conditions.
- Compare the default greedy tie-break with a `develop_balanced` action only for unseen evaluation states. This is an evaluation fallback check; it does not affect training or stored Q-values.
- Compare economic and environmental outcomes with the balanced fixed policy.

The simulation remains deterministic. Training seeds change exploration and scenario sampling; repeated evaluation under identical conditions is not an independent sample.

In [7]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from study_3_pollutant_dynamics.rl_formulation import STATE_BIN_EDGES
from study_3_pollutant_dynamics.rl_experiments import (rebuild_evaluation_trajectories, run_mixed_scenario_diagnostics,
                                                       run_training_coverage_diagnostics)

In [10]:
POLLUTION_SCENARIOS = ["low", "reference", "high"]
REWARD_MODES = ["local", "global", "mixed"]
TRAINING_SEEDS = [11, 23, 37]
EPISODE_CHECKPOINTS = [300, 1000, 3000]
EPISODE_YEARS = 40
INCOME_RETENTION = 0.6
STARTING_CURRENCY_MULTIPLIER = 1.0
DEVELOPMENT_COST_SHAPE = "constant"
EPSILON_START = 1.0
EPSILON_MINIMUM = 0.05
EPSILON_DECAY = (EPSILON_MINIMUM / EPSILON_START) ** (1 / max(EPISODE_CHECKPOINTS))
LEARNING_RATE = 0.2
DISCOUNT_FACTOR = 0.95
UNSEEN_ACTION_RULES = [None, "develop_balanced"]
RESULTS_DIRECTORY = "study_3_pollutant_dynamics/results/rl_final"
RUN_TRAINING = True
SCENARIO_RESULTS_DIRECTORY = RESULTS_DIRECTORY + "/scenario_specific"
MIXED_RESULTS_DIRECTORY = RESULTS_DIRECTORY + "/mixed_scenarios"

epsilon_summary = pd.DataFrame([{"episode": episode,
                                 "epsilon": max(EPSILON_MINIMUM, EPSILON_START * EPSILON_DECAY ** episode)}
                                for episode in EPISODE_CHECKPOINTS])
print("Epsilon decay:", round(EPSILON_DECAY, 6))
display(epsilon_summary)

Epsilon decay: 0.999002


,episode,epsilon
0,300,0.741134
1,1000,0.368403
2,3000,0.050000


## Retrain scenario-specific controls

Saved run results load by default. Set `RUN_TRAINING = True` in the setup cell to retrain. These controls were retrained with the revised epsilon schedule. That keeps the comparison with mixed-condition training matched on episode count, seed, reward mode, and exploration settings.

In [11]:
if RUN_TRAINING:
    scenario_outputs = run_training_coverage_diagnostics(
        SCENARIO_RESULTS_DIRECTORY, POLLUTION_SCENARIOS, POLLUTION_SCENARIOS, REWARD_MODES,
        TRAINING_SEEDS, EPISODE_CHECKPOINTS, STATE_BIN_EDGES, episode_years=EPISODE_YEARS,
        income_retention_fraction=INCOME_RETENTION,
        starting_currency_multiplier=STARTING_CURRENCY_MULTIPLIER,
        development_cost_shape=DEVELOPMENT_COST_SHAPE,
        epsilon_start=EPSILON_START, epsilon_decay=EPSILON_DECAY, epsilon_minimum=EPSILON_MINIMUM,
        learning_rate=LEARNING_RATE, discount_factor=DISCOUNT_FACTOR,
        unseen_action_rules=UNSEEN_ACTION_RULES,
    )
    scenario_training = pd.DataFrame(scenario_outputs[0])
    scenario_evaluation = pd.DataFrame(scenario_outputs[1])
    scenario_trajectories = pd.DataFrame(scenario_outputs[2])
    scenario_actions = pd.DataFrame(scenario_outputs[3])
    fixed_baselines = pd.DataFrame(scenario_outputs[4])
    scenario_training.to_csv(SCENARIO_RESULTS_DIRECTORY + "/training_history.csv", index=False)
    scenario_evaluation.to_csv(SCENARIO_RESULTS_DIRECTORY + "/evaluation_results.csv", index=False)
    scenario_trajectories.to_csv(SCENARIO_RESULTS_DIRECTORY + "/evaluation_trajectories.csv", index=False)
    scenario_actions.to_csv(SCENARIO_RESULTS_DIRECTORY + "/action_frequencies.csv", index=False)
    fixed_baselines.to_csv(SCENARIO_RESULTS_DIRECTORY + "/fixed_baselines.csv", index=False)
else:
    scenario_training = pd.read_csv(SCENARIO_RESULTS_DIRECTORY + "/training_history.csv")
    scenario_evaluation = pd.read_csv(SCENARIO_RESULTS_DIRECTORY + "/evaluation_results.csv")
    scenario_trajectories = pd.read_csv(SCENARIO_RESULTS_DIRECTORY + "/evaluation_trajectories.csv")
    scenario_actions = pd.read_csv(SCENARIO_RESULTS_DIRECTORY + "/action_frequencies.csv")
    fixed_baselines = pd.read_csv(SCENARIO_RESULTS_DIRECTORY + "/fixed_baselines.csv")

print("Scenario-specific training rows:", len(scenario_training))
print("Evaluation episodes:", len(scenario_evaluation))

Scenario-specific training rows: 81000
Evaluation episodes: 486


## Train policies across pollution conditions

Saved run results load by default. Set `RUN_TRAINING = True` in the setup cell to retrain. Each training episode sampled one of the three pollutant scenarios. The Q-table and epsilon schedule persist across episodes, while the environment and no-action reward reference match that episode's scenario.

In [12]:
if RUN_TRAINING:
    mixed_outputs = run_mixed_scenario_diagnostics(
        MIXED_RESULTS_DIRECTORY, POLLUTION_SCENARIOS, REWARD_MODES, TRAINING_SEEDS, EPISODE_CHECKPOINTS,
        STATE_BIN_EDGES, episode_years=EPISODE_YEARS,
        income_retention_fraction=INCOME_RETENTION,
        starting_currency_multiplier=STARTING_CURRENCY_MULTIPLIER,
        development_cost_shape=DEVELOPMENT_COST_SHAPE,
        epsilon_start=EPSILON_START, epsilon_decay=EPSILON_DECAY, epsilon_minimum=EPSILON_MINIMUM,
        learning_rate=LEARNING_RATE, discount_factor=DISCOUNT_FACTOR,
        unseen_action_rules=UNSEEN_ACTION_RULES,
    )
    mixed_training = pd.DataFrame(mixed_outputs[0])
    mixed_evaluation = pd.DataFrame(mixed_outputs[1])
    mixed_trajectories = pd.DataFrame(mixed_outputs[2])
    mixed_actions = pd.DataFrame(mixed_outputs[3])
    mixed_training.to_csv(MIXED_RESULTS_DIRECTORY + "/training_history.csv", index=False)
    mixed_evaluation.to_csv(MIXED_RESULTS_DIRECTORY + "/evaluation_results.csv", index=False)
    mixed_trajectories.to_csv(MIXED_RESULTS_DIRECTORY + "/evaluation_trajectories.csv", index=False)
    mixed_actions.to_csv(MIXED_RESULTS_DIRECTORY + "/action_frequencies.csv", index=False)
else:
    mixed_training = pd.read_csv(MIXED_RESULTS_DIRECTORY + "/training_history.csv")
    mixed_evaluation = pd.read_csv(MIXED_RESULTS_DIRECTORY + "/evaluation_results.csv")
    mixed_trajectories = pd.read_csv(MIXED_RESULTS_DIRECTORY + "/evaluation_trajectories.csv")
    mixed_actions = pd.read_csv(MIXED_RESULTS_DIRECTORY + "/action_frequencies.csv")
if not RUN_TRAINING:
    trajectory_rows = rebuild_evaluation_trajectories(
        RESULTS_DIRECTORY, scenario_evaluation.to_dict("records"), mixed_evaluation.to_dict("records"),
        episode_years=EPISODE_YEARS, income_retention_fraction=INCOME_RETENTION,
        starting_currency_multiplier=STARTING_CURRENCY_MULTIPLIER,
        development_cost_shape=DEVELOPMENT_COST_SHAPE,
    )
    tagged_trajectories = pd.DataFrame(trajectory_rows)
    scenario_trajectories = tagged_trajectories[tagged_trajectories["training_scenario"] != "mixed"].copy()
    mixed_trajectories = tagged_trajectories[tagged_trajectories["training_scenario"] == "mixed"].copy()
    scenario_trajectories.to_csv(SCENARIO_RESULTS_DIRECTORY + "/evaluation_trajectories.csv", index=False)
    mixed_trajectories.to_csv(MIXED_RESULTS_DIRECTORY + "/evaluation_trajectories.csv", index=False)
settings = {"training_scenarios": "uniform episode sampling across low, reference, high",
            "evaluation_scenarios": POLLUTION_SCENARIOS, "reward_modes": REWARD_MODES,
            "training_seeds": TRAINING_SEEDS, "episode_checkpoints": EPISODE_CHECKPOINTS,
            "episode_years": EPISODE_YEARS, "income_retention_fraction": INCOME_RETENTION,
            "development_cost_shape": DEVELOPMENT_COST_SHAPE, "epsilon_start": EPSILON_START,
            "epsilon_decay": EPSILON_DECAY, "epsilon_minimum": EPSILON_MINIMUM,
            "learning_rate": LEARNING_RATE, "discount_factor": DISCOUNT_FACTOR,
            "unseen_action_rules": ["default_greedy_tie", "develop_balanced"]}

with open(RESULTS_DIRECTORY + "/settings.json", "w", encoding="utf-8") as output_file:
    json.dump(settings, output_file, indent=2)

print("Mixed-scenario training rows:", len(mixed_training))
print("Evaluation episodes:", len(mixed_evaluation))


Mixed-scenario training rows: 27000
Evaluation episodes: 162


## Scenario exposure and state coverage

Check that the sampled training set includes all three pollution conditions, and compare unique state visits between mixed-condition and scenario-specific training.

In [13]:
exposure_rows = []
for checkpoint in EPISODE_CHECKPOINTS:
    selected_episodes = mixed_training[mixed_training["episode"] <= checkpoint]
    exposure_counts = selected_episodes["pollutant_scenario"].value_counts()
    for scenario in POLLUTION_SCENARIOS:
        exposure_rows.append({"episode_checkpoint": checkpoint, "pollutant_scenario": scenario,
                              "episodes": exposure_counts.get(scenario, 0),
                              "fraction": exposure_counts.get(scenario, 0) / checkpoint})

mixed_exposure = pd.DataFrame(exposure_rows)
display(mixed_exposure.pivot(index="episode_checkpoint", columns="pollutant_scenario", values="fraction").round(3))

pollutant_scenario,high,low,reference
episode_checkpoint,,,
300,2.980,3.140,2.880
1000,2.919,3.141,2.940
3000,2.970,3.051,2.979


In [14]:
scenario_coverage = scenario_training[scenario_training["episode"].isin(EPISODE_CHECKPOINTS)].groupby(
    ["training_scenario", "episode"], sort=False).agg(
    unique_states=("visited_states_across_regions", "mean"),
     state_action_pairs=("visited_state_action_pairs", "mean"), epsilon=("epsilon", "mean")).reset_index()

mixed_coverage = mixed_training[mixed_training["episode"].isin(EPISODE_CHECKPOINTS)].groupby("episode", sort=False).agg(
    unique_states=("visited_states_across_regions", "mean"),
     state_action_pairs=("visited_state_action_pairs", "mean"), epsilon=("epsilon", "mean")).reset_index()

mixed_coverage["training_scenario"] = "mixed"
coverage_comparison = pd.concat([scenario_coverage, mixed_coverage], ignore_index=True)
display(coverage_comparison.round(2))

,training_scenario,episode,unique_states,state_action_pairs,epsilon
0,low,300,125.22,829.44,0.74
1,low,1000,140.78,980.22,0.37
2,low,3000,149.00,1049.00,0.05
3,reference,300,185.78,1129.78,0.74
4,reference,1000,224.11,1421.11,0.37
5,reference,3000,240.22,1559.00,0.05
6,high,300,155.67,995.11,0.74
7,high,1000,180.00,1191.78,0.37
8,high,3000,193.89,1289.89,0.05
9,mixed,300,336.00,2041.44,0.74


## Matched-condition policy outcomes

This table compares policies under the pollution condition represented during training. The mixed-condition policy is tested separately under each condition. Values average the three training seeds.

In [15]:
scenario_matched = scenario_evaluation[scenario_evaluation["training_scenario"] == scenario_evaluation["evaluation_scenario"]].copy()
mixed_matched = mixed_evaluation.copy()
matched_results = pd.concat([scenario_matched, mixed_matched], ignore_index=True, sort=False)

matched_summary = matched_results.groupby(["training_scenario", "evaluation_scenario", "reward_mode", "training_episodes",
                                           "unseen_action_rule"], sort=False).agg(
    training_seeds=("training_seed", "nunique"), mean_final_EDI=("final_mean_EDI", "mean"),
     average_EDI=("average_EDI", "mean"), cumulative_net_income=("cumulative_net_income", "mean"),
     final_currency=("final_mean_currency", "mean"), development_spending=("development_spending", "mean"),
     mitigation_spending=("mitigation_spending", "mean"), final_CO2_ppm=("final_CO2_ppm", "mean"),
     final_PM2_5=("final_mean_PM2.5_ug_m3", "mean"), final_nitrate=("final_mean_nitrate_mg_N_L", "mean"),
     final_lead_total=("final_mean_lead_total_kg", "mean"), unseen_state_fraction=("unseen_state_visit_fraction", "mean")
).reset_index()

matched_results.to_csv(RESULTS_DIRECTORY + "/matched_policy_comparison.csv", index=False)
display(matched_summary[matched_summary["unseen_action_rule"] == "default_greedy_tie"].round(3))

,training_scenario,evaluation_scenario,reward_mode,training_episodes,unseen_action_rule,training_seeds,mean_final_EDI,average_EDI,cumulative_net_income,final_currency,development_spending,mitigation_spending,final_CO2_ppm,final_PM2_5,final_nitrate,final_lead_total,unseen_state_fraction
0,low,low,local,300,default_greedy_tie,3,95.750,83.098,18536.553,1906.733,3075.000,2920.000,505.394,8.569,2.265,8.071,0.073
2,low,low,local,1000,default_greedy_tie,3,97.333,83.883,18674.455,1741.168,3233.333,3506.667,505.242,8.567,2.114,7.851,0.010
4,low,low,local,3000,default_greedy_tie,3,98.250,83.985,18674.215,1681.549,3325.000,3653.333,505.154,8.567,2.083,7.892,0.000
6,low,low,global,300,default_greedy_tie,3,95.250,82.908,18540.421,1904.813,3025.000,2980.000,505.433,8.568,2.179,7.951,0.065
8,low,low,global,1000,default_greedy_tie,3,96.583,83.356,18574.057,1808.192,3158.333,3253.333,505.397,8.568,2.174,7.827,0.002
10,low,low,global,3000,default_greedy_tie,3,96.583,83.333,18600.530,1798.829,3158.333,3306.667,505.427,8.568,2.156,7.754,0.000
12,low,low,mixed,300,default_greedy_tie,3,95.750,83.050,18498.810,1877.738,3075.000,3013.333,505.443,8.569,2.256,7.877,0.073
14,low,low,mixed,1000,default_greedy_tie,3,97.333,83.671,18662.446,1701.034,3233.333,3660.000,505.266,8.567,2.096,7.649,0.010
16,low,low,mixed,3000,default_greedy_tie,3,98.000,83.715,18662.448,1647.700,3300.000,3806.667,505.181,8.566,2.040,7.652,0.000
18,reference,reference,local,300,default_greedy_tie,3,95.167,83.112,17450.885,1728.466,3016.667,3040.000,527.758,8.586,2.697,14.277,0.058


## Generalization to each pollution condition

Compare unseen-state rates and outcomes for mixed-condition policies against scenario-specific policies when the training and evaluation conditions differ.

In [16]:
generalization = pd.concat([scenario_evaluation, mixed_evaluation], ignore_index=True, sort=False)
generalization_summary = generalization[generalization["unseen_action_rule"] == "default_greedy_tie"].groupby(
    ["training_scenario", "evaluation_scenario", "reward_mode", "training_episodes"], sort=False).agg(
    training_seeds=("training_seed", "nunique"), unseen_visits=("unseen_state_visits", "mean"),
     unseen_fraction=("unseen_state_visit_fraction", "mean"), unique_unseen=("unique_unseen_states", "mean"),
     final_EDI=("final_mean_EDI", "mean"), cumulative_net_income=("cumulative_net_income", "mean"),
     mitigation_spending=("mitigation_spending", "mean"), final_CO2=("final_CO2_ppm", "mean"),
     final_PM2_5=("final_mean_PM2.5_ug_m3", "mean"), final_nitrate=("final_mean_nitrate_mg_N_L", "mean"),
     final_lead=("final_mean_lead_total_kg", "mean")).reset_index()

generalization_summary.to_csv(RESULTS_DIRECTORY + "/generalization_summary.csv", index=False)
display(generalization_summary[generalization_summary["training_episodes"] == max(EPISODE_CHECKPOINTS)].round(3))

,training_scenario,evaluation_scenario,reward_mode,training_episodes,training_seeds,unseen_visits,unseen_fraction,unique_unseen,final_EDI,cumulative_net_income,mitigation_spending,final_CO2,final_PM2_5,final_nitrate,final_lead
6,low,low,local,3000,3,0.000,0.000,0.000,98.250,18674.215,3653.333,505.154,8.567,2.083,7.892
7,low,reference,local,3000,3,97.000,0.606,20.667,78.833,17097.998,1293.333,526.996,8.582,2.587,15.542
8,low,high,local,3000,3,160.000,1.000,33.000,65.000,10988.593,0.000,576.020,8.563,2.709,56.123
15,low,low,global,3000,3,0.000,0.000,0.000,96.583,18600.530,3306.667,505.427,8.568,2.156,7.754
16,low,reference,global,3000,3,91.000,0.569,21.000,80.417,17068.413,1260.000,527.272,8.584,2.625,15.310
17,low,high,global,3000,3,160.000,1.000,33.000,65.000,10988.593,0.000,576.020,8.563,2.709,56.123
24,low,low,mixed,3000,3,0.000,0.000,0.000,98.000,18662.448,3806.667,505.181,8.566,2.040,7.652
25,low,reference,mixed,3000,3,82.000,0.513,18.333,81.417,17160.668,1873.333,527.088,8.579,2.535,15.070
26,low,high,mixed,3000,3,160.000,1.000,33.000,65.000,10988.593,0.000,576.020,8.563,2.709,56.123
33,reference,low,local,3000,3,45.333,0.283,11.667,92.167,18481.707,2786.667,505.407,8.568,2.167,8.113


## Effect of the unseen-state action rule

The alternate `develop_balanced` action is used only when the encoded evaluation state has no entry in the Q-table. Compare it with the default greedy tie-break; known-state decisions remain identical.

In [17]:
fallback_pairs = generalization.pivot_table(
    index=["policy", "training_scenario", "evaluation_scenario", "training_seed", "training_episodes", "reward_mode"],
    columns="unseen_action_rule",
    values=["final_mean_EDI", "cumulative_net_income", "mitigation_spending", "final_CO2_ppm",
            "final_mean_PM2.5_ug_m3", "final_mean_nitrate_mg_N_L", "final_mean_lead_total_kg",
            "unseen_state_visit_fraction"],
).reset_index()

fallback_pairs.columns = ["_".join(str(part) for part in column if str(part)) if isinstance(column, tuple) else column
                          for column in fallback_pairs.columns]

for metric in ["final_mean_EDI", "cumulative_net_income", "mitigation_spending", "final_CO2_ppm",
               "final_mean_PM2.5_ug_m3", "final_mean_nitrate_mg_N_L", "final_mean_lead_total_kg"]:

    fallback_pairs[metric + "_change"] = (fallback_pairs[metric + "_develop_balanced"]
                                             - fallback_pairs[metric + "_default_greedy_tie"])
fallback_summary = fallback_pairs.groupby(["training_scenario", "evaluation_scenario", "reward_mode", "training_episodes"], sort=False
).agg(mean_EDI_change=("final_mean_EDI_change", "mean"),
     mean_net_income_change=("cumulative_net_income_change", "mean"),
     mean_mitigation_change=("mitigation_spending_change", "mean"),
     mean_CO2_change=("final_CO2_ppm_change", "mean"),
     mean_PM2_5_change=("final_mean_PM2.5_ug_m3_change", "mean"),
     mean_nitrate_change=("final_mean_nitrate_mg_N_L_change", "mean"),
     mean_lead_change=("final_mean_lead_total_kg_change", "mean"),
     mean_default_unseen_fraction=("unseen_state_visit_fraction_default_greedy_tie", "mean"),
     mean_alt_unseen_fraction=("unseen_state_visit_fraction_develop_balanced", "mean")).reset_index()

display(fallback_summary[fallback_summary["training_episodes"] == max(EPISODE_CHECKPOINTS)].round(3))

,training_scenario,evaluation_scenario,reward_mode,training_episodes,mean_EDI_change,mean_net_income_change,mean_mitigation_change,mean_CO2_change,mean_PM2_5_change,mean_nitrate_change,mean_lead_change,mean_default_unseen_fraction,mean_alt_unseen_fraction
2,mixed,high,global,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,mixed,low,global,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,mixed,reference,global,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
11,mixed,high,local,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
14,mixed,low,local,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
17,mixed,reference,local,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
20,mixed,high,mixed,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
23,mixed,low,mixed,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
26,mixed,reference,mixed,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
29,high,high,global,3000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


## Fixed baseline comparison

Use the balanced rule-based policy as the primary fixed reference. The other fixed strategies remain in the saved baseline table for context.

In [18]:
balanced_baseline = fixed_baselines[fixed_baselines["policy"] == "balanced"].copy()
balanced_baseline["policy_type"] = "fixed"
balanced_baseline["training_scenario"] = balanced_baseline["evaluation_scenario"]
balanced_baseline["reward_mode"] = "fixed"
balanced_baseline["training_episodes"] = 0
balanced_baseline["unseen_action_rule"] = "not_applicable"
learned_3000 = matched_results[(matched_results["training_episodes"] == max(EPISODE_CHECKPOINTS)) &
                               (matched_results["unseen_action_rule"] == "default_greedy_tie")].copy()

baseline_comparison = pd.concat([learned_3000, balanced_baseline], ignore_index=True, sort=False)
baseline_summary = baseline_comparison.groupby(["training_scenario", "evaluation_scenario", "reward_mode", "training_episodes"],
                                               sort=False).agg(
    policy_type=("policy_type", "first"), mean_final_EDI=("final_mean_EDI", "mean"),
     average_EDI=("average_EDI", "mean"), cumulative_net_income=("cumulative_net_income", "mean"),
     final_currency=("final_mean_currency", "mean"), development_spending=("development_spending", "mean"),
     mitigation_spending=("mitigation_spending", "mean"), final_CO2=("final_CO2_ppm", "mean"),
     final_PM2_5=("final_mean_PM2.5_ug_m3", "mean"), final_nitrate=("final_mean_nitrate_mg_N_L", "mean"),
     final_lead_total=("final_mean_lead_total_kg", "mean")).reset_index()

display(baseline_summary.round(3))

,training_scenario,evaluation_scenario,reward_mode,training_episodes,policy_type,mean_final_EDI,average_EDI,cumulative_net_income,final_currency,development_spending,mitigation_spending,final_CO2,final_PM2_5,final_nitrate,final_lead_total
0,low,low,local,3000,learned,98.250,83.985,18674.215,1681.549,3325.000,3653.333,505.154,8.567,2.083,7.892
1,low,low,global,3000,learned,96.583,83.333,18600.530,1798.829,3158.333,3306.667,505.427,8.568,2.156,7.754
2,low,low,mixed,3000,learned,98.000,83.715,18662.448,1647.700,3300.000,3806.667,505.181,8.566,2.040,7.652
3,reference,reference,local,3000,learned,98.250,84.067,17595.583,1518.087,3325.000,3660.000,527.706,8.578,2.579,13.999
4,reference,reference,global,3000,learned,97.583,83.423,17427.809,1554.588,3258.333,3480.000,527.807,8.579,2.596,14.018
5,reference,reference,mixed,3000,learned,98.000,83.942,17558.065,1563.710,3300.000,3480.000,527.596,8.580,2.622,14.069
6,high,high,local,3000,learned,97.833,84.040,13972.685,996.736,3283.333,3613.333,578.063,8.651,3.110,54.868
7,high,high,global,3000,learned,95.917,83.231,13913.023,1100.703,3091.667,3353.333,578.063,8.683,3.276,55.077
8,high,high,mixed,3000,learned,97.500,83.919,13566.612,937.492,3250.000,3640.000,578.089,8.643,3.100,54.944
9,mixed,low,local,3000,learned,97.500,83.842,18657.424,1666.114,3250.000,3780.000,505.345,8.568,2.170,7.575


## Action choices and regional outcomes

Inspect whether mixed-scenario exposure changes the selected development and investment actions, and preserve regional differences in the final outcome file.

In [19]:
action_data = pd.concat([scenario_actions, mixed_actions], ignore_index=True, sort=False)
final_actions = action_data[(action_data["training_episodes"] == max(EPISODE_CHECKPOINTS)) &
                            (action_data["unseen_action_rule"] == "default_greedy_tie")]
action_summary = (final_actions.groupby(["training_scenario", "evaluation_scenario", "reward_mode", "action"], sort=False)["count"].sum()
                  .unstack(fill_value=0))
display(action_summary)

action                                             maintain  develop  \
training_scenario evaluation_scenario reward_mode                      
low               low                 local               9       88   
                                      global             20       79   
                                      mixed              17       78   
                  reference           local             291       38   
                                      global            279       69   
                                      mixed             246       47   
                  high                local             480        0   
                                      global            480        0   
                                      mixed             480        0   
reference         low                 local             138       75   
                                      global            140       74   
                                      mixed             139       93   
                  reference           local               7       96   
                                      global              6       82   
                                      mixed               6      102   
                  high                local             480        0   
                                      global            480        0   
                                      mixed             480        0   
high              low                 local             480        0   
                                      global            480        0   
                                      mixed             480        0   
                  reference           local             480        0   
                                      global            480        0   
                                      mixed             480        0   
                  high                local               6      137   
                                      global             39      104   
                                      mixed              11      131   
mixed             low                 local              15       83   
                                      global             30      106   
                                      mixed              14       96   
                  reference           local               3       94   
                                      global             12       91   
                                      mixed               5      115   
                  high                local              10      114   
                                      global             27      121   
                                      mixed              12      116   

action                                             prevention  \
training_scenario evaluation_scenario reward_mode               
low               low                 local                 2   
                                      global               14   
                                      mixed                 1   
                  reference           local                 5   
                                      global                0   
                                      mixed                 0   
                  high                local                 0   
                                      global                0   
                                      mixed                 0   
reference         low                 local                 2   
                                      global               14   
                                      mixed                 2   
                  reference           local                 1   
                                      global               41   
                                      mixed                 5   
                  high                local                 0   
                                      global                0   
                            

In [20]:
regional_source = pd.concat([scenario_trajectories, mixed_trajectories], ignore_index=True, sort=False)

regional_outcomes = regional_source[regional_source["step"] == EPISODE_YEARS].groupby([
    "policy_type", "policy", "training_scenario",                                                                               "evaluation_scenario", "training_seed", "training_episodes", "reward_mode", "unseen_action_rule", "region"], dropna=False, sort=False).agg(
    final_EDI=("EDI", "mean"), final_currency=("currency", "mean"), cumulative_net_income=("net_income", "sum"),
     development_spending=("development_spending", "sum"), mitigation_spending=("mitigation_spending", "sum"),
     mean_PM2_5=("PM2.5_ug_m3", "mean"), CO2=("CO2_ppm", "mean"), nitrate=("nitrate_mg_N_L", "mean"),
     lead_total=("lead_total_kg", "mean")).reset_index()

regional_outcomes.to_csv(RESULTS_DIRECTORY + "/regional_final_outcomes.csv", index=False)
display(regional_outcomes.head(24).round(3))

,policy_type,policy,training_scenario,evaluation_scenario,training_seed,training_episodes,reward_mode,unseen_action_rule,region,final_EDI,final_currency,cumulative_net_income,development_spending,mitigation_spending,mean_PM2_5,CO2,nitrate,lead_total
0,fixed,no_development_no_investment,low,low,NaN,NaN,NaN,not_applicable,Northeast,70.0,3223.570,135.373,0.0,0.0,7.420,503.220,1.376,9.472
1,fixed,no_development_no_investment,low,low,NaN,NaN,NaN,not_applicable,Midwest,60.0,3040.981,133.882,0.0,0.0,8.079,503.220,2.408,9.632
2,fixed,no_development_no_investment,low,low,NaN,NaN,NaN,not_applicable,South,55.0,2976.523,136.009,0.0,0.0,8.145,503.220,2.064,9.512
3,fixed,no_development_no_investment,low,low,NaN,NaN,NaN,not_applicable,West,75.0,3337.436,135.805,0.0,0.0,10.608,503.220,1.376,9.432
4,fixed,development_only,low,low,NaN,NaN,NaN,not_applicable,Northeast,100.0,2755.543,155.138,0.0,0.0,7.428,507.262,1.824,9.544
5,fixed,development_only,low,low,NaN,NaN,NaN,not_applicable,Midwest,100.0,2079.595,125.174,25.0,0.0,8.094,507.262,3.576,9.760
6,fixed,development_only,low,low,NaN,NaN,NaN,not_applicable,South,95.0,2157.728,141.005,25.0,0.0,8.160,507.262,3.150,9.619
7,fixed,development_only,low,low,NaN,NaN,NaN,not_applicable,West,100.0,2992.755,155.646,0.0,0.0,10.614,507.262,1.728,9.488
8,fixed,prevention_focused,low,low,NaN,NaN,NaN,not_applicable,Northeast,100.0,2124.744,156.519,0.0,0.0,7.420,504.349,1.459,9.439
9,fixed,prevention_focused,low,low,NaN,NaN,NaN,not_applicable,Midwest,100.0,1667.378,148.630,25.0,0.0,8.082,504.349,2.860,9.601


## Findings

- Epsilon stayed at 0.742 at episode 300 and 0.369 at 1,000, reaching 0.05 at 3,000. The mixed run sampled low, reference, and high conditions at close to one-third each.
- At 3,000 episodes, mixed-condition Q-tables contained about 440 to 453 distinct states across four regions, compared with about 145 to 240 for scenario-specific training. Mixed-trained policies had zero unseen-state visits in each of the three evaluation conditions.
- Scenario-specific policies had zero unseen states when evaluated in their own training condition, but cross-condition unseen-state visits reached 29% to 100%. Under the default tie-break, unseen high-pollution states selected `maintain` and mean EDI remained 65. Mixed training removed that no-action failure mode.
- From 300 to 3,000 episodes, mixed-trained final EDI improved by 0.5 to 2.75 points, depending on reward mode and pollution condition. At 3,000 episodes, mean final EDI ranged from 95.67 to 98.08 across the mixed policies and evaluation conditions. Scenario-specific matched EDI was similar, so mixed training mainly improved generalization rather than matched-condition performance.
- The balanced fixed policy still reached 98.75 EDI in each condition and spent 6,400 on mitigation. Mixed-trained policies generally spent about 3,100 to 3,800 and did not dominate the fixed policy across income, EDI, spending, and pollution outcomes.
- The alternate `develop_balanced` action changed scenario-specific cross-condition results sharply because it replaced the default `maintain` choice at unseen states. In fully unseen cases it added 33.75 mean EDI and 6,400 mitigation spending. Mixed-trained policies had no unseen-state visits, so the fallback did not change their outcomes.

Mixed-condition training meaningfully improves state coverage and generalization, enough to continue with tabular Q-Learning for focused policy analysis. Increasing episode count alone is not the main opportunity. Remaining limits are partial observability of project timing and lagged income/damage, the large sparse state space, and the gap from the balanced fixed baseline. Keep the reward and environment equations unchanged while investigating those limits.
